In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ====== 3. Install Python Packages ======
!pip install torch torchvision torchaudio --upgrade  # PyTorch
!pip install opencv-python matplotlib numpy pycocotools psutil Pillow
!pip install tqdm

# ====== 4. Verify GPU Availability (if using CUDA) ======
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [3]:
import sys
sys.path.append('/content/drive/MyDrive/CV Project 2025/segmentation')

In [4]:
import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


from coco_utils import get_coco, collate_fn
from engine import train_one_epoch, evaluate

In [5]:
def get_instance_segmentation_model(num_classes):
    # load an instance segmentation model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model


In [6]:
img_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/images'
train_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/train.json'
val_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/val.json'
test_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/test.json'

In [7]:
coco_train_ds = get_coco(img_data_path, train_nn_data_path, mode="instances", train=True)
coco_val_ds = get_coco(img_data_path, val_nn_data_path, mode="instances", train=False)
coco_test_ds = get_coco(img_data_path, test_nn_data_path, mode="instances", train=False)

loading annotations into memory...
Done (t=0.27s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


In [8]:
train_data_loader = torch.utils.data.DataLoader(coco_train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)

val_data_loader = torch.utils.data.DataLoader(coco_val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

test_data_loader = torch.utils.data.DataLoader(coco_test_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [9]:
num_classes = 2

# get the model using our helper function
model = get_instance_segmentation_model(num_classes)
# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# and a learning rate scheduler which decreases the learning rate by
# 10x every 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# let's train it for 10 epochs
from torch.optim.lr_scheduler import StepLR

In [10]:
num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, train_data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # # evaluate on the test dataset
    evaluate(model, val_data_loader, device=device)

    torch.save(
      {
          "epoch": epoch,
          "model_state_dict": model.state_dict(),
          "optimizer_state_dict": optimizer.state_dict(),
      },
      "maskrcnn_same_object.pt",
    )

/content/drive/MyDrive/CV Project 2025/segmentation/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/580]  eta: 1:12:12  lr: 0.000014  loss: 7.8195 (7.8195)  loss_classifier: 0.5314 (0.5314)  loss_box_reg: 0.3958 (0.3958)  loss_mask: 6.8581 (6.8581)  loss_objectness: 0.0200 (0.0200)  loss_rpn_box_reg: 0.0142 (0.0142)  time: 7.4698  data: 2.8649  max mem: 4328
Epoch: [0]  [ 10/580]  eta: 0:23:23  lr: 0.000100  loss: 5.4252 (4.8471)  loss_classifier: 0.4682 (0.4697)  loss_box_reg: 0.3958 (0.4425)  loss_mask: 4.7116 (3.8276)  loss_objectness: 0.0541 (0.0856)  loss_rpn_box_reg: 0.0142 (0.0217)  time: 2.4620  data: 0.7401  max mem: 4587
Epoch: [0]  [ 20/580]  eta: 0:18:38  lr: 0.000186  loss: 1.8782 (3.2649)  loss_classifier: 0.3388 (0.3654)  loss_box_reg: 0.3395 (0.3796)  loss_mask: 1.0834 (2.4291)  loss_objectness: 0.0421 (0.0729)  loss_rpn_box_reg: 0.0083 (0.0180)  time: 1.7232  data: 0.4116  max mem: 4587
Epoch: [0]  [ 30/580]  eta: 0:16:44  lr: 0.000272  loss: 1.2936 (2.5043)  loss_classifier: 0.2093 (0.3120)  loss_box_reg: 0.2714 (0.3587)  loss_mask: 0.4421 (1.7544) 

In [24]:
# download the model
from google.colab import files
files.download("maskrcnn_same_object.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes

from PIL import Image, ImageOps
import PIL
import numpy as np

def save_result(img, int_img, prediction):
    img = Image.fromarray(img.mul(255).permute(1, 2, 0).byte().numpy())

    output = prediction[0]
    masks, boxes = output["masks"], output["boxes"]

    detection_threshold = 0.8
    pred_scores = output["scores"].detach().cpu().numpy()
    pred_classes = [str(i) for i in output["labels"].cpu().numpy()]
    pred_bboxes = output["boxes"].detach().cpu().numpy()
    boxes = pred_bboxes[pred_scores >= detection_threshold].astype(np.int32)
    pred_classes = pred_classes[: len(boxes)]

    int_img = np.array(int_img)
    int_img = np.transpose(int_img, [2, 0, 1])
    int_img = torch.tensor(int_img, dtype=torch.uint8)

    colors = np.random.randint(0, 255, size=(len(pred_bboxes), 3))
    colors = [tuple(color) for color in colors]
    result_with_boxes = draw_bounding_boxes(
        int_img,
        boxes=torch.tensor(boxes),
        width=4,
        colors=colors,
        labels=pred_classes,
    )

    final_masks = masks > 0.5
    final_masks = final_masks.squeeze(1)
    seg_result = draw_segmentation_masks(
        result_with_boxes, final_masks, colors=colors, alpha=0.8
    )

    seg_img = Image.fromarray(seg_result.mul(255).permute(1, 2, 0).byte().numpy())

    imgs = [img, seg_img]
    min_shape = sorted([(np.sum(i.size), i.size) for i in imgs])[0][1]
    imgs_comb = np.hstack([i.resize(min_shape) for i in imgs])

    imgs_comb = Image.fromarray(imgs_comb)
    imgs_comb.save("result.jpg")

In [26]:
# test the model on the test set

evaluate(model, test_data_loader, device=device)

# randomly sample one index from the test set
sample_idx = torch.randint(len(coco_test_ds), size=(1,)).item()
int_img, img, _ = coco_test_ds[sample_idx]
model.eval()
with torch.no_grad():
  prediction = model([img.to(device)])

Test:  [  0/507]  eta: 0:07:15  model_time: 0.1711 (0.1711)  evaluator_time: 0.0197 (0.0197)  time: 0.8593  data: 0.6163  max mem: 4946
Test:  [100/507]  eta: 0:02:47  model_time: 0.1218 (0.1747)  evaluator_time: 0.0715 (0.1434)  time: 0.5085  data: 0.0571  max mem: 4946
Test:  [200/507]  eta: 0:02:06  model_time: 0.1327 (0.1790)  evaluator_time: 0.0684 (0.1425)  time: 0.4801  data: 0.0584  max mem: 4946
Test:  [300/507]  eta: 0:01:23  model_time: 0.1361 (0.1769)  evaluator_time: 0.0526 (0.1346)  time: 0.4278  data: 0.0575  max mem: 4946
Test:  [400/507]  eta: 0:00:44  model_time: 0.1645 (0.1819)  evaluator_time: 0.1038 (0.1394)  time: 0.4681  data: 0.0669  max mem: 4946
Test:  [500/507]  eta: 0:00:02  model_time: 0.1549 (0.1839)  evaluator_time: 0.0717 (0.1405)  time: 0.4912  data: 0.0663  max mem: 4946
Test:  [506/507]  eta: 0:00:00  model_time: 0.1370 (0.1833)  evaluator_time: 0.0351 (0.1396)  time: 0.4112  data: 0.0547  max mem: 4946
Test: Total time: 0:03:30 (0.4151 s / it)
Averag

In [20]:
save_result(img, int_img, prediction)

In [25]:
# download the model
from google.colab import files
files.download("result.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>